# Create Multimap Sequential Dataset

This notebook iterates through all 14 of the raw `.npz` datasets, processes them into sequences grouped by episode, and saves the final list of all episodes as a single `.pkl` file.

This is the recommended format for training an LSTM, as it preserves the temporal sequence of each episode while handling variable lengths.

In [6]:
import numpy as np
from pathlib import Path
import pickle
import os

# 1. Locate project root and dataset run directory
project_root = Path.cwd().parent
base_data_dir = project_root / "mpc_datasets" / "run_20260409T193833Z"
print(f"Project root: {project_root}")
print(f"Base data dir: {base_data_dir}")

# 2. List of all map + direction combinations to process
map_names = [
    "Spielberg_normal", "Spielberg_reverse",
    "Budapest_normal", "Budapest_reverse",
    "Monza_normal", "Monza_reverse",
    "Spa_normal", "Spa_reverse",
    "Silverstone_normal", "Silverstone_reverse",
    "Melbourne_normal", "Melbourne_reverse",
    "Montreal_normal", "Montreal_reverse",
]
print(f"Processing {len(map_names)} map configs")

# 3. Aggregate all episodes across all maps
all_episode_sequences = []

total_steps = 0
for map_name in map_names:
    map_folder, direction = map_name.split("_")
    npz_path = base_data_dir / map_folder / direction / f"kmpc_{map_name}.npz"

    if not npz_path.exists():
        print(f"[WARN] Missing file for {map_name}: {npz_path}")
        continue

    data = np.load(npz_path)

    # Drop x,y from observations: keep [delta, linear_vel_x, pose_theta]
    observations = data["observations"][:, 2:]  # (N, 3)
    lidar_scans = data["lidar_scans"]          # (N, 60)
    inputs = np.concatenate([observations, lidar_scans], axis=1)  # (N, 63)

    # Use expert_actions (clean MPC actions) as targets
    targets = data["expert_actions"]  # (N, 2)

    episode_ids = data["episode_ids"]
    unique_eps = np.unique(episode_ids)

    collision_flags = data["collision_flags"] if "collision_flags" in data.files else None
    boundary_flags = data["boundary_flags"] if "boundary_flags" in data.files else None

    for ep_id in unique_eps:
        mask = episode_ids == ep_id
        ep_inputs = inputs[mask]
        ep_targets = targets[mask]

        # Episode-level collision/boundary indicators
        has_collision = False
        has_boundary = False
        if collision_flags is not None:
            coll_ep = collision_flags[mask]
            has_collision = bool(coll_ep.any()) if coll_ep.size > 0 else False
        if boundary_flags is not None:
            bound_ep = boundary_flags[mask]
            has_boundary = bool(bound_ep.any()) if bound_ep.size > 0 else False

        all_episode_sequences.append({
            "inputs": ep_inputs,
            "targets": ep_targets,
            "map": map_name,
            "episode_id_local": int(ep_id),
            "has_collision": has_collision,
            "has_boundary": has_boundary,
        })

        total_steps += ep_inputs.shape[0]

    print(f"{map_name}: {len(unique_eps)} episodes, {inputs.shape[0]} steps")

print(f"\nTotal episodes aggregated (including collisions): {len(all_episode_sequences)}")
print(f"Total steps aggregated: {total_steps}")


Project root: /home/devin_work/work/f1tenth/ApproxiMPC
Base data dir: /home/devin_work/work/f1tenth/ApproxiMPC/mpc_datasets/run_20260409T193833Z
Processing 14 map configs
Spielberg_normal: 10 episodes, 87428 steps
Spielberg_reverse: 10 episodes, 87246 steps
Budapest_normal: 10 episodes, 103076 steps
Budapest_reverse: 10 episodes, 102933 steps
Monza_normal: 10 episodes, 113335 steps
Monza_reverse: 10 episodes, 12034 steps
Spa_normal: 10 episodes, 140699 steps
Spa_reverse: 10 episodes, 140656 steps
Silverstone_normal: 10 episodes, 116624 steps
Silverstone_reverse: 10 episodes, 116673 steps
Melbourne_normal: 10 episodes, 121160 steps
Melbourne_reverse: 10 episodes, 121365 steps
Montreal_normal: 10 episodes, 74693 steps
Montreal_reverse: 10 episodes, 75106 steps

Total episodes aggregated (including collisions): 140
Total steps aggregated: 1413028


In [10]:
# Helper: inspect episode length distribution before deciding on padding

# Option: drop collision episodes for this analysis as well
non_collision_episodes = [ep for ep in all_episode_sequences if not ep.get("has_collision", False)]

if len(all_episode_sequences) == 0:
    print("No episodes found. Run the aggregation cell above first.")
else:
    print(f"Total episodes (raw):        {len(all_episode_sequences)}")
    print(f"Episodes without collisions: {len(non_collision_episodes)}")
    print(f"Episodes with collisions:    {len(all_episode_sequences) - len(non_collision_episodes)}")

    if len(non_collision_episodes) == 0:
        print("No collision-free episodes to analyze.")
    else:
        # Compute lengths (number of steps) for each collision-free episode
        episode_lengths = [ep["inputs"].shape[0] for ep in non_collision_episodes]
        lengths_arr = np.asarray(episode_lengths)
        print("\nLength stats for collision-free episodes only:")
        print(f"  Min steps per episode:    {lengths_arr.min()}")
        print(f"  Max steps per episode:    {lengths_arr.max()}")
        print(f"  Mean steps per episode:   {lengths_arr.mean():.1f}")
        print(f"  Median steps per episode: {np.median(lengths_arr):.1f}")

        # Optional: rough histogram of lengths
        bins = [0, 6000, 7000, 8000, 9000, 10000, 11000, 12000, 13000, 14000, 15000]
        hist, edges = np.histogram(lengths_arr, bins=bins)
        print("\nEpisode length histogram (steps, collision-free only):")
        for count, left, right in zip(hist, edges[:-1], edges[1:]):
            print(f"  [{left:4d}, {right:4d}): {count} episodes")


Total episodes (raw):        140
Episodes without collisions: 131
Episodes with collisions:    9

Length stats for collision-free episodes only:
  Min steps per episode:    7450
  Max steps per episode:    14093
  Mean steps per episode:   10781.0
  Median steps per episode: 11327.0

Episode length histogram (steps, collision-free only):
  [   0, 6000): 0 episodes
  [6000, 7000): 0 episodes
  [7000, 8000): 20 episodes
  [8000, 9000): 20 episodes
  [9000, 10000): 0 episodes
  [10000, 11000): 20 episodes
  [11000, 12000): 31 episodes
  [12000, 13000): 20 episodes
  [13000, 14000): 0 episodes
  [14000, 15000): 20 episodes


In [8]:
# Helper: inspect why very short episodes occur (collisions, boundaries, etc.)

# Configure how many of the shortest episodes to inspect
num_short_episodes_to_show = 10

if len(all_episode_sequences) == 0:
    print("No episodes found. Run the aggregation cell first.")
else:
    # 1. Build a list of (idx, length, map, episode_id_local)
    epi_info = [
        (idx, ep["inputs"].shape[0], ep["map"], ep["episode_id_local"])
        for idx, ep in enumerate(all_episode_sequences)
    ]

    # 2. Sort by length ascending
    epi_info_sorted = sorted(epi_info, key=lambda x: x[1])

    # 3. Take the shortest few
    to_inspect = epi_info_sorted[:num_short_episodes_to_show]

    print(f"Inspecting the {len(to_inspect)} shortest episodes:")

    for global_idx, length, map_name, ep_id in to_inspect:
        map_folder, direction = map_name.split("_")
        npz_path = base_data_dir / map_folder / direction / f"kmpc_{map_name}.npz"

        if not npz_path.exists():
            print(f"\nEpisode {global_idx} ({map_name}, ep_id={ep_id}, len={length}): data file missing at {npz_path}")
            continue

        data = np.load(npz_path)
        episode_ids = data["episode_ids"]
        mask = (episode_ids == ep_id)

        # Pull per-step flags for just this episode
        collision = data["collision_flags"][mask]
        boundary = data["boundary_flags"][mask]
        term = data["terminations"][mask]
        trunc = data["truncations"][mask]

        any_collision = bool(collision.any()) if collision.size > 0 else False
        any_boundary = bool(boundary.any()) if boundary.size > 0 else False
        any_term = bool(term.any()) if term.size > 0 else False
        any_trunc = bool(trunc.any()) if trunc.size > 0 else False

        # Look at last step flags to guess end reason
        end_reason = "step_limit_or_unknown"
        if term.size > 0 or trunc.size > 0:
            last_term = bool(term[-1])
            last_trunc = bool(trunc[-1])
            last_collision = bool(collision[-1]) if collision.size > 0 else False
            last_boundary = bool(boundary[-1]) if boundary.size > 0 else False

            if last_term:
                if last_collision:
                    end_reason = "collision_terminated"
                elif last_boundary:
                    end_reason = "boundary_terminated"
                else:
                    end_reason = "env_terminated"
            elif last_trunc:
                end_reason = "env_truncated"

        print(f"\nEpisode {global_idx}: map={map_name}, local_ep_id={ep_id}, steps={length}")
        print(f"  any_collision: {any_collision}, any_boundary: {any_boundary}")
        print(f"  any_terminated: {any_term}, any_truncated: {any_trunc}")
        print(f"  inferred_end_reason: {end_reason}")

Inspecting the 10 shortest episodes:

Episode 52: map=Monza_reverse, local_ep_id=2, steps=79
  any_collision: True, any_boundary: True
  any_terminated: True, any_truncated: False
  inferred_end_reason: boundary_terminated

Episode 53: map=Monza_reverse, local_ep_id=3, steps=79
  any_collision: True, any_boundary: True
  any_terminated: True, any_truncated: False
  inferred_end_reason: boundary_terminated

Episode 54: map=Monza_reverse, local_ep_id=4, steps=79
  any_collision: True, any_boundary: True
  any_terminated: True, any_truncated: False
  inferred_end_reason: boundary_terminated

Episode 56: map=Monza_reverse, local_ep_id=6, steps=79
  any_collision: True, any_boundary: True
  any_terminated: True, any_truncated: False
  inferred_end_reason: boundary_terminated

Episode 57: map=Monza_reverse, local_ep_id=7, steps=79
  any_collision: True, any_boundary: True
  any_terminated: True, any_truncated: False
  inferred_end_reason: boundary_terminated

Episode 59: map=Monza_reverse, l

In [9]:
# 4. Save list-of-episodes structure as a pickle file
# For this first training set, we exclude any episodes that experienced a collision.

non_collision_episodes = [ep for ep in all_episode_sequences if not ep.get("has_collision", False)]

output_dir = project_root / "datasets"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "multimap_lstm_sequences_no_collisions.pkl"

with open(output_path, "wb") as f:
    pickle.dump(non_collision_episodes, f)

size_mb = os.path.getsize(output_path) / 1e6
print(f"Saved PKL dataset (no-collision episodes only) to: {output_path} ({size_mb:.2f} MB)")
print(f"Episodes saved: {len(non_collision_episodes)} of {len(all_episode_sequences)} total")

Saved PKL dataset (no-collision episodes only) to: /home/devin_work/work/f1tenth/ApproxiMPC/datasets/multimap_lstm_sequences_no_collisions.pkl (367.22 MB)
Episodes saved: 131 of 140 total
